# CEG-WM Content V4 — clean-null whitening fit

Initial user-only GPU handoff for 32 clean images. Create the Colab Secret `HF_TOKEN`, then run the cells once from top to bottom. Stop after any failure. The final cell only validates and downloads an existing exact-bound asset pair.

## 1. Fresh exact checkout

In [ ]:
import json
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
BRANCH = "stage-a-content-adaptive-dual-branch-v4-whitened-lf"
EXACT = "79f67646595bd99cc8b066cad0e4b12e96a22cbb"
RUNNER_MODULE = "experiments.run_content_v4_clean_null_whitening_fit"
ASSET_FILENAME = "content_v4_clean_null_whitening_operator_v1.json"
RECEIPT_PREFIX = "CEGWM_CONTENT_V4_WHITENING_RECEIPT"
FAILURE_PREFIX = "CEGWM_CONTENT_V4_WHITENING_HANDOFF_FAILURE"

repo = pathlib.Path("/content/cegwm-stage-a-content-v4-whitening-fit-source")
local_root = pathlib.Path("/content/cegwm-stage-a-content-v4-whitening-fit-local")
artifact_sink = pathlib.Path("/content/drive/MyDrive/CEG-WM/content_v4_whitening_fit")
bound_result_dir = artifact_sink / EXACT
asset_path = bound_result_dir / ASSET_FILENAME
checksum_path = bound_result_dir / (ASSET_FILENAME + ".sha256")

HANDOFF_FAILED = False
RUNNER_ATTEMPTED = False

def fail(stage):
    global HANDOFF_FAILED
    if HANDOFF_FAILED:
        return
    HANDOFF_FAILED = True
    payload = {"status": "operational_failure", "producer_exact": EXACT, "stage": stage}
    print(FAILURE_PREFIX + " " + json.dumps(payload, sort_keys=True, separators=(",", ":")), flush=True)

def git(*args):
    return subprocess.run(
        ["git", *args], cwd=repo, check=True, capture_output=True, text=True
    ).stdout.strip()

try:
    if repo.exists():
        raise FileExistsError
    subprocess.run(
        ["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(repo)],
        check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    if (
        git("branch", "--show-current") != BRANCH
        or git("rev-parse", "HEAD") != EXACT
        or git("status", "--porcelain")
    ):
        raise RuntimeError
except BaseException:
    fail("source_checkout")

## 2. Install the checked-out project

In [ ]:
if not HANDOFF_FAILED:
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", str(repo)],
            check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        if (
            git("branch", "--show-current") != BRANCH
            or git("rev-parse", "HEAD") != EXACT
            or git("status", "--porcelain")
        ):
            raise RuntimeError
    except BaseException:
        fail("dependency_install")

## 3. Run once with bounded output

This cell mounts Drive, prepares the task-local Hugging Face cache, passes `HF_TOKEN` only to the child process, and prints either one validated receipt or one sanitized failure line.

In [ ]:
import os
import re
from google.colab import drive, userdata

if not HANDOFF_FAILED and not RUNNER_ATTEMPTED:
    RUNNER_ATTEMPTED = True
    process = None
    runner_env = None
    hf_token = ""
    captured = bytearray()
    capture_overflow = False
    runner_rc = None
    launch_failed = False
    CAPTURE_LIMIT = 4096
    try:
        drive.mount("/content/drive")
        if local_root.exists() or bound_result_dir.exists() or asset_path.exists() or checksum_path.exists():
            raise FileExistsError
        local_root.mkdir(parents=True, exist_ok=False)
        hf_cache = local_root / "hf-cache"
        hf_cache.mkdir(exist_ok=False)
        hf_token = userdata.get("HF_TOKEN")
        if not isinstance(hf_token, str) or not hf_token.strip():
            raise RuntimeError
        if (
            git("branch", "--show-current") != BRANCH
            or git("rev-parse", "HEAD") != EXACT
            or git("status", "--porcelain")
            or bound_result_dir.exists()
        ):
            raise RuntimeError
        secret_markers = ("TOKEN", "KEY", "SECRET", "PASSWORD", "CREDENTIAL")
        runner_env = {
            name: value for name, value in os.environ.items()
            if not any(marker in name.upper() for marker in secret_markers)
        }
        runner_env["HF_TOKEN"] = hf_token
        runner_env["HF_HOME"] = str(hf_cache)
        hf_token = ""
        process = subprocess.Popen(
            [
                sys.executable, "-m", RUNNER_MODULE,
                "--repo-root", str(repo),
                "--expected-exact", EXACT,
                "--artifact-sink", str(artifact_sink),
            ],
            cwd=repo, env=runner_env, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL,
        )
        while True:
            chunk = process.stdout.read(1024)
            if not chunk:
                break
            remaining = max(0, CAPTURE_LIMIT - len(captured))
            captured.extend(chunk[:remaining])
            if len(chunk) > remaining:
                capture_overflow = True
        runner_rc = process.wait()
    except BaseException:
        launch_failed = True
        if process is not None and process.poll() is None:
            process.kill()
            process.wait()
    finally:
        hf_token = ""
        if runner_env is not None:
            runner_env.pop("HF_TOKEN", None)
            runner_env.pop("HF_HOME", None)
        runner_env = None

    if launch_failed:
        captured.clear()
        fail("fit_runner_launch")
    elif runner_rc != 0:
        captured.clear()
        fail("fit_runner_nonzero")
    elif capture_overflow:
        captured.clear()
        fail("fit_runner_stdout_overflow")
    else:
        try:
            captured_text = captured.decode("utf-8", errors="strict")
            if captured_text.count("\n") != 1 or not captured_text.endswith("\n"):
                raise RuntimeError
            line = captured_text[:-1]
            if not line.startswith(RECEIPT_PREFIX + " "):
                raise RuntimeError
            receipt = json.loads(line.split(" ", 1)[1])
            if (
                not isinstance(receipt, dict)
                or set(receipt) != {"asset_sha256", "producer_exact", "unit_count"}
                or re.fullmatch(r"[0-9a-f]{64}", receipt["asset_sha256"]) is None
                or receipt["producer_exact"] != EXACT
                or receipt["unit_count"] != 32
            ):
                raise RuntimeError
            print(line, flush=True)
        except BaseException:
            fail("fit_runner_receipt")
        finally:
            captured.clear()


## 4. Validate and download the existing pair

This runner-free cell is usable after a successful top-to-bottom run or by itself after reconnecting. It stays silent when an earlier cell recorded a failure.

In [ ]:
import hashlib
import json
import pathlib
import re
from google.colab import drive, files

EXACT = "79f67646595bd99cc8b066cad0e4b12e96a22cbb"
ASSET_FILENAME = "content_v4_clean_null_whitening_operator_v1.json"
FAILURE_PREFIX = "CEGWM_CONTENT_V4_WHITENING_HANDOFF_FAILURE"
artifact_sink = pathlib.Path("/content/drive/MyDrive/CEG-WM/content_v4_whitening_fit")
result_dir = artifact_sink / EXACT
asset_path = result_dir / ASSET_FILENAME
checksum_path = result_dir / (ASSET_FILENAME + ".sha256")

if not bool(globals().get("HANDOFF_FAILED", False)):
    artifact_failed = False

    def artifact_fail(stage):
        global artifact_failed
        if artifact_failed:
            return
        artifact_failed = True
        payload = {"status": "operational_failure", "producer_exact": EXACT, "stage": stage}
        print(FAILURE_PREFIX + " " + json.dumps(payload, sort_keys=True, separators=(",", ":")), flush=True)

    try:
        if not pathlib.Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
        expected_names = sorted([ASSET_FILENAME, ASSET_FILENAME + ".sha256"])
        if (
            not result_dir.is_dir()
            or not asset_path.is_file()
            or not checksum_path.is_file()
            or sorted(path.name for path in result_dir.iterdir()) != expected_names
        ):
            raise RuntimeError
        asset_bytes = asset_path.read_bytes()
        sidecar = checksum_path.read_text(encoding="ascii")
        if re.fullmatch(r"[0-9a-f]{64}  " + re.escape(ASSET_FILENAME) + r"\n", sidecar) is None:
            raise RuntimeError
        if hashlib.sha256(asset_bytes).hexdigest() != sidecar[:64]:
            raise RuntimeError
        asset_bytes = b""
    except BaseException:
        artifact_fail("existing_asset_pair_validation")

    if not artifact_failed:
        try:
            files.download(str(asset_path))
            files.download(str(checksum_path))
        except BaseException:
            artifact_fail("existing_asset_pair_download")